In [1]:
# Load packages
suppressPackageStartupMessages({
    library(Seurat)
    library(scran)
    library(scater)
    library(batchelor)
    library(SingleCellExperiment)
    library(scater)
    library(Matrix)
    library(reshape2)
    library(data.table)
    library(BiocParallel)
    library(dplyr)
    library(gridExtra)
})

    ncores = 4
    mcparam = MulticoreParam(workers = ncores)
    register(mcparam)
    BPPARAM = SerialParam()

options(repr.plot.width=15, repr.plot.height=8)

In [2]:
main = "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/"
source(paste0(main, "mapping_functions_extended.R"))

In [3]:
normalise = TRUE
remove_doublets = FALSE
remove_stripped = FALSE
load_corrected = FALSE
  
  # This function was taken from 
  # https://github.com/MarioniLab/EmbryoTimecourse2018/blob/master/analysis_scripts/atlas/core_functions.R
  # with minor adaptations
  
  if(load_corrected & (!remove_doublets | !remove_stripped)){
    message("Using corrected PCs, also removing doublets + stripped now.")
    remove_doublets = TRUE
    remove_stripped = TRUE
  }
  
  atlas_in = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/'
  counts = readMM(paste0(atlas_in, 'raw_counts.mtx')) # readMM for .mtx input 
  genes = read.table(paste0(atlas_in, 'genes.tsv'), stringsAsFactors = F)
  meta = read.table(paste0(atlas_in, 'meta.tab'), header = TRUE, sep = "\t", stringsAsFactors = FALSE, comment.char = "$")
  
  rownames(counts) = genes[,1] #ensembl
  colnames(counts) = meta$cell
  
  sce = SingleCellExperiment(assays = list("counts" = counts))
  
  if(normalise){
    sfs = read.table(paste0(atlas_in, 'sizefactors.tab'), stringsAsFactors = F)[,1]
    sizeFactors(sce) = sfs
    sce = logNormCounts(sce)
  }
  
  if(remove_doublets){
    sce = logNormCounts(sce[,!meta$doublet])
    meta = meta[!meta$doublet,]
  }
  
  if(remove_stripped){
    sce = logNormCounts(sce[,!meta$stripped])
    meta = meta[!meta$stripped, ]
  }
  
  if(load_corrected){
    corrected = readRDS(paste0(atlas_in, "corrected_pcas.rds"))
    assign("corrected", corrected, envir = .GlobalEnv)
    }

In [5]:
head(meta,2)
sce

,cell,sample,stage,celltype_original,celltype
,<chr>,<int>,<chr>,<chr>,<chr>
1,cell_1,1,E6.5,Epiblast,Epiblast
2,cell_2,1,E6.5,Primitive Streak,Primitive Streak


class: SingleCellExperiment 
dim: 23972 115559 
metadata(0):
assays(2): counts logcounts
rownames(23972): ENSMUSG00000001138 ENSMUSG00000001143 ...
  ENSMUSG00000108929 ENSMUSG00000109022
rowData names(0):
colnames(115559): cell_1 cell_2 ... ext_cell_334312 ext_cell_311352
colData names(1): sizeFactor
reducedDimNames(0):
altExpNames(0):

In [ ]:
#get order: oldest to youngest; most cells to least cells
hvgs = getHVGs(sce, block = meta$sample)

  message("Batch effect correction for the atlas...")  
  order_df        <- meta[!duplicated(meta$sample), c("stage", "sample")]
  order_df$ncells <- sapply(order_df$sample, function(x) sum(atlas_meta$sample == x))
  order_df$stage  <- factor(order_df$stage, 
                            levels = rev(c("E9.5",
                                           "E9.25",
                                           "E9.0",
                                           "E8.75",
                                           "E8.5",
                                           "E8.25",
                                           "E8.0",
                                           "E7.75",
                                           "E7.5",
                                           "E7.25",
                                           "mixed_gastrulation",
                                           "E7.0",
                                           "E6.75",
                                           "E6.5")))
  order_df       <- order_df[order(order_df$stage, order_df$ncells, decreasing = TRUE),]
  order_df$stage <- as.character(order_df$stage)
  

all_correct = doBatchCorrect(counts = logcounts(sce)[rownames(sce) %in% hvgs,], 
                             timepoints = meta$stage, 
                             samples = meta$sample, 
                             timepoint_order = order_df$stage, 
                             sample_order = order_df$sample, 
                             npc = 30)

Batch effect correction for the atlas...



In [3]:
sce
head(meta)
head(genes)

class: SingleCellExperiment 
dim: 23972 115559 
metadata(0):
assays(2): counts logcounts
rownames(23972): ENSMUSG00000001138 ENSMUSG00000001143 ...
  ENSMUSG00000108929 ENSMUSG00000109022
rowData names(0):
colnames(115559): cell_1 cell_2 ... ext_cell_334312 ext_cell_311352
colData names(1): sizeFactor
reducedDimNames(0):
altExpNames(0):

,cell,sample,stage,celltype_original,celltype
,<chr>,<int>,<chr>,<chr>,<chr>
1,cell_1,1,E6.5,Epiblast,Epiblast
2,cell_2,1,E6.5,Primitive Streak,Primitive Streak
3,cell_5,1,E6.5,ExE ectoderm,ExE ectoderm
4,cell_6,1,E6.5,Epiblast,Epiblast
5,cell_8,1,E6.5,Epiblast,Epiblast
6,cell_9,1,E6.5,Epiblast,Epiblast


,V1
,<chr>
1,ENSMUSG00000001138
2,ENSMUSG00000001143
3,ENSMUSG00000001674
4,ENSMUSG00000002459
5,ENSMUSG00000002881
6,ENSMUSG00000003134
